# Research Report Builder | Plan-and-Execute (Planner-Executor)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, List, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

class PlanExecuteState(TypedDict):
    task: str
    plan: List[str]
    current_step: NotRequired[int]
    step_results: List[str]
    final_output: NotRequired[str]

def parse_json(text: str):
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    return json.loads(cleaned)

In [4]:
def create_plan(state: PlanExecuteState) -> dict:
    """Create a detailed execution plan for the task."""
    response = model.invoke(
        f"Create a step-by-step plan to complete this task. Each step should be concrete and actionable.\n\n"
        f"Task: {state['task']}\n\n"
        f"Return a JSON list of step descriptions (3-5 steps). Example:\n"
        f'["Research X", "Analyze Y", "Synthesize findings into Z"]'
    )
    try:
        plan = parse_json(response.content)
        # Handle both list and dict responses
        if isinstance(plan, dict):
            plan = plan.get("steps", plan.get("subtasks", list(plan.values())[0]))
        if not isinstance(plan, list):
            plan = [str(plan)]
    except (json.JSONDecodeError, TypeError, IndexError):
        # Fallback: split response into lines as steps
        plan = [s.strip() for s in response.content.split("\n") if s.strip()]
    return {"plan": plan, "current_step": 0, "step_results": []}

In [5]:
def execute_step(state: PlanExecuteState) -> Command[Literal["execute_step", "synthesize"]]:
    """Execute the current step in the plan."""
    step_idx = state.get("current_step", 0)
    plan = state.get("plan", [])
    if not plan or step_idx >= len(plan):
        return Command(goto="synthesize")

    current_task = plan[step_idx]
    previous_context = ""
    if state.get("step_results"):
        previous_context = "\n\nPrevious step results:\n" + "\n---\n".join(
            f"Step {i+1}: {state['plan'][i]}\nResult: {r}"
            for i, r in enumerate(state["step_results"])
        )

    response = model.invoke(
        f"You are executing step {step_idx + 1} of a plan.\n\n"
        f"Overall task: {state['task']}\n"
        f"Current step: {current_task}\n"
        f"{previous_context}\n\n"
        f"Complete this step thoroughly. Provide detailed output."
    )
    new_results = list(state.get("step_results", [])) + [response.content]
    new_step = step_idx + 1
    if new_step >= len(plan):
        return Command(goto="synthesize", update={"step_results": new_results, "current_step": new_step})
    return Command(goto="execute_step", update={"step_results": new_results, "current_step": new_step})

In [6]:
def synthesize(state: PlanExecuteState) -> dict:
    """Combine all step results into a final output."""
    all_results = "\n\n---\n\n".join(
        f"## Step {i+1}: {state['plan'][i]}\n\n{result}"
        for i, result in enumerate(state["step_results"])
    )
    response = model.invoke(
        f"Synthesize the following research into a comprehensive, well-structured report.\n\n"
        f"Task: {state['task']}\n\n"
        f"Step Results:\n{all_results}"
    )
    return {"final_output": response.content}

In [7]:
# Build graph
graph = StateGraph(PlanExecuteState)
graph.add_node("create_plan", create_plan)
graph.add_node("execute_step", execute_step, destinations=("execute_step", "synthesize"))
graph.add_node("synthesize", synthesize)

graph.add_edge(START, "create_plan")
graph.add_edge("create_plan", "execute_step")
# execute_step returns Command to route directly
graph.add_edge("synthesize", END)

planner = graph.compile()

In [8]:
plot_mermaid(planner)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	create_plan(create_plan)
	execute_step(execute_step)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> create_plan;
	create_plan --> execute_step;
	execute_step -.-> synthesize;
	synthesize --> __end__;
	execute_step -.-> execute_step;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = planner.invoke({
    "task": "Create a competitive analysis of the top 3 cloud providers (AWS, Azure, GCP) focusing on AI/ML services"
})
print(result["final_output"])

# Competitive Analysis Report: AI/ML Services of AWS, Azure, and GCP

## Executive Summary

This report provides a detailed competitive analysis of the AI/ML services offered by Amazon Web Services (AWS), Microsoft Azure, and Google Cloud Platform (GCP). It highlights the unique strengths, weaknesses, opportunities, and threats of each provider, aiming to help organizations make informed decisions when choosing a cloud service for AI/ML applications.

## Introduction

Cloud computing has become a cornerstone for deploying AI/ML solutions due to its scalability, flexibility, and cost-effectiveness. Among the leading cloud service providers, AWS, Azure, and GCP stand out for their comprehensive offerings in AI/ML services. This report synthesizes recent data on their service capabilities, ease of use, integration, scalability, and performance.

## AI/ML Services Overview

### Amazon Web Services (AWS)

#### Services and Tools
AWS offers an extensive suite of AI/ML services, notably Amazo

In [10]:
# Streaming

stream_invoke(
    planner, {
        "task": "Create a competitive analysis of the top 3 cloud providers (AWS, Azure, GCP) focusing on AI/ML services"
    }
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'task': 'Create a competitive analysis of the top 3 cloud providers (AWS, Azure, GCP) focusing on AI/ML services',
 'plan': ['Research the AI/ML services offered by AWS, Azure, and GCP. Gather information on specific services, tools, pricing, and features from official websites and other reputable sources.',
  'Compare the AI/ML service offerings of each provider. Analyze characteristics such as ease of use, integration capabilities, scalability, and performance benchmarks.',
  "Investigate case studies, customer reviews, and expert opinions to evaluate the real-world effectiveness and adoption of each cloud provider's AI/ML services.",
  'Identify key differentiators and competitive advantages for each provider by synthesizing findings from research and analysis.',
  "Compile the findings into a comprehensive report, highlighting the strengths, weaknesses, opportunities, and threats (SWOT analysis) of each provider's AI/ML services."],
 'current_step': 5,
 'step_results': ["To conduc